# Finnhub - Khám phá sàn chứng khoán FinHub API

Notebook này giúp bạn khám phá nhanh thư viện `finnhub-python`:
- Kiểm tra kết nối API qua biến môi trường
- Kiểm tra dữ liệu real-time qua websocket
- Kiểm tra những sàn giao dịch hỗ trợ
- Lấy dữ liệu nến từ quá khứ
- Lấy tin tức chung của sàn chứng khoán

## 1. Cấu hình

In [11]:
import sys
import os
import asyncio
import pandas as pd
import finnhub
from datetime import datetime
from IPython.display import clear_output, display

# Thêm thư mục gốc vào path để import src
sys.path.append(os.path.abspath('..'))

from src.config import Config
from src.streamer import finnhub_streamer

In [2]:
# Kiểm tra key
if not Config.FINNHUB_API_KEY:
    print("Error: FINNHUB_API_KEY is not set in the environment variables.")
    sys.exit(1)
else:
    print("FINNHUB_API_KEY: ", Config.FINNHUB_API_KEY[0:4] + "...")  # In một phần của key để xác nhận

FINNHUB_API_KEY:  d7gs...


In [41]:
finnhub_client = finnhub.Client(api_key=Config.FINNHUB_API_KEY)

## 2. Giao dịch - Cập nhật giá cuối cùng

Finhub có hỗ trợ cho bản free Websocket để lấy giá liên tục từ chứng khoán thị trường Hoa Kỳ

In [13]:
symbols_to_track = ["AAPL", "BINANCE:BTCUSDT", "OANDA:EUR_USD"]
api_key = Config.FINNHUB_API_KEY 

In [ ]:
import time
from collections import defaultdict
# ===== INIT =====
tick_window = defaultdict(list)       # symbol -> [monotonic timestamps trong 1s gần nhất]
max_tick_per_symbol = defaultdict(int)
max_total_ticks = 0

latest_prices = {
    symbol: {
        "price": "Đang đợi...",
        "volume": "0",
        "time": "--:--:--",
        "ticks": 0,
        "max_ticks": 0
    }
    for symbol in symbols_to_track
}

WINDOW_SIZE = 1.0  # Rolling window 1 giây thực


# ===== MAIN TRACKER =====
async def run_tracker():
    global max_total_ticks

    print("🚀 Đang kết nối server Finnhub...")

    try:
        async for trades in finnhub_streamer(api_key, symbols_to_track):
            now = time.monotonic()  # Wall-clock thực, không bị ảnh hưởng bởi timestamp server

            for trade in trades:
                symbol = trade['s']

                # Ghi nhận thời điểm nhận trade
                tick_window[symbol].append(now)

                # Xóa các ticks cũ hơn 1 giây (rolling window)
                tick_window[symbol] = [
                    t for t in tick_window[symbol]
                    if now - t <= WINDOW_SIZE
                ]

                current_ticks = len(tick_window[symbol])

                # Cập nhật max per symbol
                if current_ticks > max_tick_per_symbol[symbol]:
                    max_tick_per_symbol[symbol] = current_ticks

                # Dùng trade timestamp chỉ để hiển thị
                dt_object = datetime.fromtimestamp(trade['t'] / 1000.0)

                latest_prices[symbol] = {
                    "price": f"{trade['p']:,.2f}",
                    "volume": f"{trade['v']:.4f}",
                    "time": dt_object.strftime('%H:%M:%S'),
                    "ticks": current_ticks,
                    "max_ticks": max_tick_per_symbol[symbol]
                }

            # Tính total ticks từ rolling window thực (cùng thời điểm now)
            total_ticks = sum(
                len([t for t in tick_window[sym] if now - t <= WINDOW_SIZE])
                for sym in symbols_to_track
            )

            if total_ticks > max_total_ticks:
                max_total_ticks = total_ticks

            # ===== DISPLAY =====
            clear_output(wait=True)

            print(f"📊 BẢNG GIÁ REAL-TIME ({datetime.now().strftime('%d/%m/%Y %H:%M:%S')})")
            print(f"🔥 Now: {total_ticks} ticks/s | 🚀 Peak: {max_total_ticks} ticks/s")
            print("-" * 95)
            print(f"{'Mã':<18} | {'Giá (USD)':>15} | {'KL':>15} | {'Ticks/s':>8} | {'Max':>6} | {'Thời gian'}")
            print("-" * 95)

            for sym in symbols_to_track:
                info = latest_prices[sym]
                print(
                    f"{sym:<18} | "
                    f"{info['price']:>15} | "
                    f"{info['volume']:>15} | "
                    f"{info['ticks']:>8} | "
                    f"{info['max_ticks']:>6} | "
                    f"{info['time']}"
                )

            print("-" * 95)
            print("💡 Mẹo: Nhấn Stop (Interrupt) để dừng.")

    except asyncio.CancelledError:
        print("\n⏹️ Đã dừng Task thành công.")
    except Exception as e:
        print(f"\n❌ Có lỗi xảy ra: {e}")


# ===== RUN =====
stream_task = asyncio.create_task(run_tracker())

📊 BẢNG GIÁ REAL-TIME (19/04/2026 13:17:06)
🔥 Now: 178 ticks/s | 🚀 Peak: 1801 ticks/s
-----------------------------------------------------------------------------------------------
Mã                 |       Giá (USD) |              KL |  Ticks/s |    Max | Thời gian
-----------------------------------------------------------------------------------------------
AAPL               |     Đang đợi... |               0 |        0 |      0 | --:--:--
BINANCE:BTCUSDT    |       75,347.08 |          0.0132 |      178 |   1801 | 13:17:05
OANDA:EUR_USD      |            1.18 |          0.0000 |        1 |      1 | 03:59:05
-----------------------------------------------------------------------------------------------
💡 Mẹo: Nhấn Stop (Interrupt) để dừng.

⏹️ Đã dừng Task thành công.


Nếu muốn dừng, bạn chạy lệnh dưới đây

In [15]:
stream_task.cancel() # Dừng task nếu cần thiết

True

Hiện tại dữ liệu được lấy liên tục từ finnhub không được đúng theo trình tự, có một vài trading (phần lớn từ crypto) có độ trễ rơi và khoảng 2-4 phút mới tới nơi. Nên nến sẽ được tính min là 5 phút để giảm việc cập nhật liên tục và rơi rớt dữ liệu

## 3. Một vài sàn giao dịch hỗ trợ

Dưới đây là danh sách các sàn giao dịch của crypto

In [ ]:
print(finnhub_client.crypto_exchanges())

['BINANCE', 'POLONIEX', 'OKEX', 'BINANCEUS', 'COINBASE', 'GEMINI', 'HUOBI', 'KRAKEN', 'HITBTC', 'KUCOIN', 'BITFINEX', 'BITMEX']


Tra cứu symbols theo các sàn giao dịch  
Ví dụ: BINANCE

In [40]:
crypto = 'BINANCE'
crypto_symbols = pd.DataFrame(finnhub_client.crypto_symbols(crypto)).head(10)
crypto_symbols

,description,displaySymbol,symbol
0,Binance NULS/BUSD,NULS/BUSD,BINANCE:NULSBUSD
1,Binance DNT/USDT,DNT/USDT,BINANCE:DNTUSDT
2,Binance C/BNB,C/BNB,BINANCE:CBNB
3,Binance SLF/BTC,SLF/BTC,BINANCE:SLFBTC
4,Binance KLAY/BTC,KLAY/BTC,BINANCE:KLAYBTC
5,Binance AVAX/GBP,AVAX/GBP,BINANCE:AVAXGBP
6,Binance CHZ/USDT,CHZ/USDT,BINANCE:CHZUSDT
7,Binance METIS/USDT,METIS/USDT,BINANCE:METISUSDT
8,Binance VANA/FDUSD,VANA/FDUSD,BINANCE:VANAFDUSD
9,Binance DOGS/BNB,DOGS/BNB,BINANCE:DOGSBNB


Tra cứu biểu tượng, phần này để bạn có thể tra cứu theo keyword

In [ ]:
key = 'apple'
symbol_look = finnhub_client.symbol_lookup(key)

print("Tổng số lượng:", symbol_look['count'])
display(pd.DataFrame(symbol_look['result']))

Tổng số lượng: 11


,description,displaySymbol,symbol,type
0,Apple Inc,AAPL,AAPL,Common Stock
1,Apple Hospitality REIT Inc,APLE,APLE,Common Stock
2,Apple Flavor & Fragrance Group Co Ltd,603020.SS,603020.SS,Common Stock
3,Maui Land & Pineapple Company Inc,MLP,MLP,Common Stock
4,Apple iSports Group Inc,AAPI,AAPI,Common Stock
5,Apple International Co Ltd,2788.T,2788.T,Common Stock
6,Applepark Co Ltd,164A.T,164A.T,Common Stock
7,Pineapple Financial Inc,PAPL,PAPL,Common Stock
8,Pineapple Resources Bhd,PINEAPP.KL,PINEAPP.KL,Common Stock
9,Pineapple Energy Inc,SUNE,SUNE,Common Stock


Phần này sẽ liệt kê cổ phiếu được hỗ trợ, có thể truy cập link dưới đây để tìm thấy cổ phiếu trên Finnhub, hiện tại api free chỉ lấy được US
- https://docs.google.com/spreadsheets/d/1I3pBxjfXB056-g_JYf_6o3Rns3BV2kMGG1nCatb91ls/edit?gid=0#gid=0

In [ ]:
stock_symbols = pd.DataFrame(finnhub_client.stock_symbols('US'))
stock_symbols

,currency,description,displaySymbol,figi,figiComposite,isin,mic,shareClassFIGI,symbol,symbol2,type
0,USD,OJAI OIL CO,OJOC,BBG000BHYP24,BBG000BHYP24,,OOTC,BBG001S7S572,OJOC,,Common Stock
1,USD,TEMA DURABLE QUALITY ETF,TOLL,BBG01GL6QNW2,BBG01GL6QNW2,,BATS,BBG01GL6QPR3,TOLL,,ETP
2,USD,VICTORIAS MILLING CO INC,VMCIF,BBG01XQ6D0Y2,BBG01XQ6D0Y2,,OOTC,BBG001S7M355,VMCIF,,Common Stock
3,USD,RESERVE PETROLEUM COMPANY,RSRV,BBG000BR6GT4,BBG000BR6GT4,,OOTC,BBG001S6XY66,RSRV,,Common Stock
4,USD,ENERTOPIA CORP,ENRT,BBG000PRDZ46,BBG000PRDZ46,,OOTC,BBG001S970K7,ENRT,,Common Stock
...,...,...,...,...,...,...,...,...,...,...,...
30270,USD,TAKUMA CO LTD,TKUMF,BBG000CNGTJ4,BBG000CNGTJ4,,OOTC,BBG001S6FH18,TKUMF,,Common Stock
30271,USD,NUVEEN ESG MID-CAP GROW ETF,NUMG,BBG00FJ5HK56,BBG00FJ5HK56,,BATS,BBG00FJ5HKX5,NUMG,,ETP
30272,USD,ALAMOS GOLD INC-CLASS A,AGI,BBG009HT6BL4,BBG009HT6BL4,,XNYS,BBG009HT65S0,AGI,,Common Stock
30273,USD,BASEL MEDICAL GROUP LTD,BMGL,BBG01PR70N34,BBG01PR70N34,,XNAS,BBG01PR70NY0,BMGL,,Common Stock


In [ ]:
print("Tổng số symbols US:", len(stock_symbols))
print(stock_symbols['symbol'].tolist())

Tổng số symbols US: 30275
['OJOC', 'TOLL', 'VMCIF', 'RSRV', 'ENRT', 'ECAOF', 'SEBYF', 'GXUSF', 'JXAMY', 'RNAOS', 'REFI', 'AVK', 'YOOV', 'SBOEY', 'TAFI', 'PUCKW', 'CLWY', 'RAFE', 'MYRLF', 'CARS', 'ACWX', 'BDVSF', 'LFSWF', 'BTZRF', 'EMA', 'HLFBF', 'AUMBF', 'CSGNF', 'IRDM', 'HUABF', 'CNONF', 'RFHRF', 'KGS', 'DVXY', 'RSBRS', 'MRWES', 'AIRI', 'PDMI', 'PWP', 'MNMXF', 'PCWLF', 'TLGRF', 'NEON', 'GSGTF', 'LRRLF', 'ANPDF', 'NXEN', 'VEDL', 'QBCAF', 'CISS', 'RISN', 'AIRJW', 'ODSMF', 'SNPTF', 'ORXCF', 'AEXA', 'GEF', 'IHETW', 'PRCWF', 'UL', 'PMIFF', 'KUYAF', 'FTNJ', 'FASDF', 'BFAP', 'MALG', 'TMMI', 'DMA', 'LCNTU', 'RNNCF', 'HYGV', 'XVIPF', 'OGGNF', 'CRACW', 'ANCMF', 'BTG', 'EPYFF', 'NJMVF', 'MSKCS', 'USRI', 'LMND', 'MZZ', 'VWOB', 'QNICF', 'TLFX', 'SHUS', 'RPKIF', 'TEUUF', 'VAGPF', 'SIRLF', 'CYPRY', 'FMUN', 'AIFF', 'NMMRF', 'PIKL', 'BIEI', 'LQD', 'TJIPF', 'RSLBF', 'TRTX', 'BOWNU', 'AMFPY', 'AEF', 'RNKGF', 'LII', 'RQI', 'FSGCY', 'HLEGF', 'TLXPF', 'RSBMS', 'RTBBF', 'FBDHF', 'GEMG', 'SPGI', 'NOEMW', 'FL

## 4. Dữ liệu nến từ quá khứ

Do finnhub đã khóa lấy dữ liệu nến quá khứ cho free, để bù đắp chuyện này, chúng ta sẽ sử dụng thư viện yfinance để bù lại và nối đuôi với finnhub thực tế sau